## **ENTRENAMIENTO SAM 2 CON ISIC-2016**

En este fichero se evalúa el rendimiento de los modleos SAM 2 Base y SAM 2 Large tras haberlos entrenado usando el dataset ISIC-2016.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import torch
import shutil
import random
import torch.nn as nn


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, resize_for_hausdorff, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_fine_tuning


En primer lugar, se definen las rutas al dataset original y a la carpeta de salida donde se guardará la nueva estructura.

ISIC 2016 ya viene con una división train/test oficial. Por tanto, la función split_dataset usa las imágenes de test originales y subdivide el conjunto de entrenamiento original en train (85%) y val (15%) con semilla fija para garantizar reproducibilidad. Una vez definidos los splits, copia las imágenes y las máscaras a sus carpetas correspondientes, renombrando las máscaras para eliminar el sufijo "_Segmentation".

In [ ]:
DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"
OUTPUT_ROOT  = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"

TRAIN_IMAGES = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data")
TRAIN_MASKS  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth")
TEST_IMAGES  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data")
TEST_MASKS   = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth")

def split_dataset(train_ratio=0.85):
    train_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TRAIN_MASKS) if f.endswith("_Segmentation.png")
    ]
    test_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TEST_MASKS) if f.endswith("_Segmentation.png")
    ]

    random.seed(42)
    random.shuffle(train_names)
    n_train = int(len(train_names) * train_ratio)

    splits = {
        "train": [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[:n_train]],
        "val":   [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[n_train:]],
        "test":  [(n, TEST_IMAGES,  TEST_MASKS)  for n in test_names],
    }

    for split, entries in splits.items():
        os.makedirs(os.path.join(OUTPUT_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, "masks",  split), exist_ok=True)
        for name, img_src_dir, mask_src_dir in entries:
            shutil.copy(
                os.path.join(img_src_dir,  name + ".jpg"),
                os.path.join(OUTPUT_ROOT, "images", split, name + ".jpg")
            )
            shutil.copy(
                os.path.join(mask_src_dir, name + "_Segmentation.png"),
                os.path.join(OUTPUT_ROOT, "masks",  split, name + ".png")
            )

split_dataset()

En primer lugar, se habilita TF32 para las operaciones matriciales, ya que esto nos permite acelerar el entrenamiento en GPUs NVIDIA modernas con una pérdida de precisión mínima. Además, se define el dispositivo donde se ejecutará el entrenamiento y se comprueba la disponibilidad de CUDA.

A continuación, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test). También, se filtran únicamente las muestras para las que existe el fichero de máscara.

Por último, en el método __getitem__ se aplica el preprocesamiento de cada muestra. La imagen se redimensiona a 1024x1024, se normaliza entre 0 y 1 y se convierte a tensor. Por otro lado, la máscara se redimensiona a 256x256 (resolución de salida del mask decoder de SAM) y se binariza. Finalmente, el punto central se calcula a partir de los contornos de la máscara con cv2.findContours y se obtiene la caja delimitadora que los engloba. A partir de dicha caja delimitadora se calcula el centro escalado a las nuevas dimensiones de la imagen.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_filename in os.listdir(self.images_dir):
            if not img_filename.lower().endswith(".jpg"):
                continue
            name      = img_filename.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_filename)
            mask_path = os.path.join(self.masks_dir,  name + ".png")
            if os.path.exists(mask_path):
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image    = cv2.imread(img_path)
        image    = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        image    = cv2.resize(image, (self.img_size, self.img_size))
        image    = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask     = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask     = cv2.resize(mask, (256, 256))
        mask     = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        gt_full  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_bin   = (gt_full > 127).astype(np.uint8)
        contours, _ = cv2.findContours(gt_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        x, y, w, h  = cv2.boundingRect(np.vstack(contours))

        cx = (x + w / 2) * (self.img_size / orig_w)
        cy = (y + h / 2) * (self.img_size / orig_h)

        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


Esta función contiene el bucle de entrenamiento. En primer lugar, se carga el modelo SAM 2 mediante build_sam2, que requiere además de los pesos un fichero de configuración específico de la arquitectura. Al igual que con SAM 1, se congelan tanto el image encoder como el prompt encoder, de forma que durante el entrenamiento solo se actualizan los pesos del mask decoder.

Como optimizador se usa Adam con un learning rate de 1e-4, y se añade un scheduler que reduce el learning rate a la mitad cada 20 épocas, ayudando así a afinar el modelo conforme avanza el entrenamiento. Además, se usa BCEWithLogitsLoss como función de pérdida, ya que es adecuada para segmentación binaria.

Por último, al usar la API de SAM 2, el image encoder se ejecuta a través de predictor.set_image, lo que permite acceder a las características de alta resolución (high_res_feats) que SAM 2 incorpora y que se pasan directamente al mask decoder. Tanto el image encoder como el prompt encoder se ejecutan sin calcular gradientes, mientras que el mask decoder sí los calcula. La pérdida se promedia sobre el batch, se hace backpropagation, se actualizan los pesos y al final de cada época el scheduler ajusta el learning rate. Finalmente, el guardado de pesos se hace en formato de diccionario con la clave "model", ya que es el formato esperado por la API de SAM 2.

In [ ]:
def train_sam(dataset_path, weights_path, config_path, output_weights, epochs=50, batch_size=4):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)

    for param in sam2.image_encoder.parameters():
        param.requires_grad = False
    for param in sam2.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam2.sam_mask_decoder.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam2.train()
    predictor = SAM2ImagePredictor(sam2)

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):
                image_np = (images[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)

                with torch.no_grad():
                    predictor.set_image(image_np)

                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam2.sam_prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                image_embedding   = predictor._features["image_embed"]
                image_pe          = sam2.sam_prompt_encoder.get_dense_pe()
                high_res_features = predictor._features["high_res_feats"]

                low_res_masks, _, _, _ = sam2.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save({"model": sam2.state_dict()}, output_weights)
    return output_weights


**SAM 2 BASE ENTRENAMIENTO**

En primer lugar, se definen las rutas a los distintos recursos que se van a usar: el dataset, los pesos preentrenados, el fichero de configuración de la arquitectura y las rutas donde se guardarán los resultados y los pesos del modelo entrenado. En este caso, se procede a evaluar el rendimiento del modelo SAM 2 Base.

Se define también la función get_central_point, que calcula el punto central de una máscara detectando sus contornos y obteniendo el centro de la caja delimitadora que los engloba, devolviendo None si la máscara está vacía.

Respecto a la función de evaluación, esta evalúa el modelo fine-tuneado sobre el conjunto de test. Para cada imagen se calcula el punto central mediante la función get_central_point, descartando las imágenes cuya máscara esté vacía. La máscara predicha se redimensiona al tamaño original de la imagen para que sea comparable con el ground truth, y se calculan todas las métricas. Al final se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Finalmente, se lanza el entrenamiento midiendo el tiempo que tarda, y después se evalúa el modelo resultante, midiendo también su tiempo. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores, lo que permite ir acumulando los resultados de distintos experimentos en el mismo fichero.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

DATASET_ROOT   = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"
WEIGHTS        = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt"
CONFIG         = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2\\sam2_hiera_b+.yaml"
OUTPUT_WEIGHTS = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2b_isic2016.pt"
OUTPUT_PATH    = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def get_central_point(mask_bin):
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return [[x + w / 2, y + h / 2]]


def evaluate_model(dataset_path, weights_path, config_path):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    for img_filename in sorted(os.listdir(test_images_dir)):
        if not img_filename.lower().endswith(".jpg"):
            continue

        name      = img_filename.replace(".jpg", "")
        img_path  = os.path.join(test_images_dir, img_filename)
        mask_path = os.path.join(test_masks_dir,  name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)
        if gt_mask.sum() == 0:
            continue

        central_point = get_central_point(gt_mask)
        if central_point is None:
            continue

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference_fine_tuning(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo":              ["sam2_b_isic2016"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    return results


start_train = time.time()
trained_weights = train_sam(DATASET_ROOT, WEIGHTS, CONFIG, OUTPUT_WEIGHTS)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(DATASET_ROOT, trained_weights, CONFIG)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
if os.path.exists(OUTPUT_PATH):
    df.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
else:
    df.to_csv(OUTPUT_PATH, index=False)


Epoch 1/50 - Loss: 0.1464
Epoch 2/50 - Loss: 0.1049
Epoch 3/50 - Loss: 0.1010
Epoch 4/50 - Loss: 0.0937
Epoch 5/50 - Loss: 0.0848
Epoch 6/50 - Loss: 0.0831
Epoch 7/50 - Loss: 0.0786
Epoch 8/50 - Loss: 0.0768
Epoch 9/50 - Loss: 0.0732
Epoch 10/50 - Loss: 0.0756
Epoch 11/50 - Loss: 0.0735
Epoch 12/50 - Loss: 0.0689
Epoch 13/50 - Loss: 0.0712
Epoch 14/50 - Loss: 0.0705
Epoch 15/50 - Loss: 0.0638
Epoch 16/50 - Loss: 0.0599
Epoch 17/50 - Loss: 0.0595
Epoch 18/50 - Loss: 0.0561
Epoch 19/50 - Loss: 0.0578
Epoch 20/50 - Loss: 0.0537
Epoch 21/50 - Loss: 0.0494
Epoch 22/50 - Loss: 0.0445
Epoch 23/50 - Loss: 0.0419
Epoch 24/50 - Loss: 0.0410
Epoch 25/50 - Loss: 0.0402
Epoch 26/50 - Loss: 0.0393
Epoch 27/50 - Loss: 0.0386
Epoch 28/50 - Loss: 0.0385
Epoch 29/50 - Loss: 0.0394
Epoch 30/50 - Loss: 0.0386
Epoch 31/50 - Loss: 0.0376
Epoch 32/50 - Loss: 0.0364
Epoch 33/50 - Loss: 0.0354
Epoch 34/50 - Loss: 0.0354
Epoch 35/50 - Loss: 0.0346
Epoch 36/50 - Loss: 0.0350
Epoch 37/50 - Loss: 0.0370
Epoch 38/5

**SAM 2 LARGE ENTRENAMIENTO**

A continuación, se replica el procedimiento aplicado en el bloque de código anterior con la única diferencia de que el modelo a evaluar en este caso será SAM 2 Large. De esta manera podremos comparar las diferencias de rendimiento entre las distintas versiones de cada modelo.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

DATASET_ROOT   = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"
WEIGHTS        = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt"
CONFIG         = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2\\sam2_hiera_l.yaml"
OUTPUT_WEIGHTS = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2l_isic2016.pt"
OUTPUT_PATH    = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def get_central_point(mask_bin):
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return [[x + w / 2, y + h / 2]]


def evaluate_model(dataset_path, weights_path, config_path):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    for img_filename in sorted(os.listdir(test_images_dir)):
        if not img_filename.lower().endswith(".jpg"):
            continue

        name      = img_filename.replace(".jpg", "")
        img_path  = os.path.join(test_images_dir, img_filename)
        mask_path = os.path.join(test_masks_dir,  name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)
        if gt_mask.sum() == 0:
            continue

        central_point = get_central_point(gt_mask)
        if central_point is None:
            continue

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference_fine_tuning(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue

        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo":              ["sam2_l_isic2016"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    return results


start_train = time.time()
trained_weights = train_sam(DATASET_ROOT, WEIGHTS, CONFIG, OUTPUT_WEIGHTS)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(DATASET_ROOT, trained_weights, CONFIG)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
if os.path.exists(OUTPUT_PATH):
    df.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
else:
    df.to_csv(OUTPUT_PATH, index=False)


Epoch 1/50 - Loss: 0.1487
Epoch 2/50 - Loss: 0.1084
Epoch 3/50 - Loss: 0.0993
Epoch 4/50 - Loss: 0.0910
Epoch 5/50 - Loss: 0.0869
Epoch 6/50 - Loss: 0.0824
Epoch 7/50 - Loss: 0.0820
Epoch 8/50 - Loss: 0.0756
Epoch 9/50 - Loss: 0.0746
Epoch 10/50 - Loss: 0.0710
Epoch 11/50 - Loss: 0.0687
Epoch 12/50 - Loss: 0.0813
Epoch 13/50 - Loss: 0.0676
Epoch 14/50 - Loss: 0.0652
Epoch 15/50 - Loss: 0.0621
Epoch 16/50 - Loss: 0.0592
Epoch 17/50 - Loss: 0.0577
Epoch 18/50 - Loss: 0.0555
Epoch 19/50 - Loss: 0.0549
Epoch 20/50 - Loss: 0.0539
Epoch 21/50 - Loss: 0.0482
Epoch 22/50 - Loss: 0.0441
Epoch 23/50 - Loss: 0.0415
Epoch 24/50 - Loss: 0.0409
Epoch 25/50 - Loss: 0.0396
Epoch 26/50 - Loss: 0.0388
Epoch 27/50 - Loss: 0.0387
Epoch 28/50 - Loss: 0.0403
Epoch 29/50 - Loss: 0.0390
Epoch 30/50 - Loss: 0.0389
Epoch 31/50 - Loss: 0.0375
Epoch 32/50 - Loss: 0.0363
Epoch 33/50 - Loss: 0.0357
Epoch 34/50 - Loss: 0.0351
Epoch 35/50 - Loss: 0.0349
Epoch 36/50 - Loss: 0.0346
Epoch 37/50 - Loss: 0.0333
Epoch 38/5